# Standalone PDelta3-GDN2-CLVR — Decision Model

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_PDelta3_GDN2_Standalone_Decision_Colab.ipynb)

Standalone successor to `Laya_PDelta3_GDN2_CLVR_Colab.ipynb` for later game/control use.

- **No `laya` Python dependency** in checkpoint reconstruction, conversion, training, export, reload, or inference.
- Replaces **every ModernBERT `full_attention` encoder layer**.
- Also replaces all full `nn.MultiheadAttention` layers in the typed-decision head.
- Keeps ModernBERT's already-local sliding attention unchanged.
- PDelta3 uses GDN2 + bidirectional global linear memory + Local32 + local convolution/direct-value routes; no global dense T×T softmax attention remains.
- Runs per-layer transfer warm-up plus a short end-to-end decision distillation pass.
- Exports the **entire standalone model**, not an adapter, then reloads it through `runtime.decide(state, actions)`.


In [ ]:
#@title 1. Install
%pip -q install -U "transformers>=4.45" "datasets>=2.20" "huggingface_hub>=0.25" "safetensors>=0.4"
%pip -q install "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@codex/pdelta3-standalone-decision"


In [ ]:
#@title 2. Configure conversion and short fine-training
import json, os, sys, torch
from tinycenn_lm.standalone_pdelta3_training import PDelta3TrainConfig, train_and_export, upload_export
from tinycenn_lm.standalone_pdelta3_decision import load_standalone_pdelta3

SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
FEATURE_DIM = 96 #@param {type:"integer"}
LOCAL_WINDOW = 32 #@param {type:"integer"}
ENCODER_STEPS = 120 #@param {type:"integer"}
HEAD_STEPS = 80 #@param {type:"integer"}
JOINT_STEPS = 300 #@param {type:"integer"}
TRAIN_CASES = 900 #@param {type:"integer"}
VAL_CASES = 200 #@param {type:"integer"}
MAX_LEN = 384 #@param {type:"integer"}
BATCH_SIZE = 3 #@param {type:"integer"}
OUTPUT_DIR = "/content/PDelta3_GDN2_Standalone_Decision" #@param {type:"string"}

cfg = PDelta3TrainConfig(
    source_model=SOURCE_MODEL,
    feature_dim=FEATURE_DIM,
    local_window=LOCAL_WINDOW,
    encoder_steps=ENCODER_STEPS,
    head_steps=HEAD_STEPS,
    joint_steps=JOINT_STEPS,
    train_cases=TRAIN_CASES,
    val_cases=VAL_CASES,
    max_len=MAX_LEN,
    batch_size=BATCH_SIZE,
    output_dir=OUTPUT_DIR,
)
assert "laya" not in sys.modules
cfg


In [ ]:
#@title 3. Convert ALL full-attention layers + stabilize + evaluate + export
result = train_and_export(cfg)

print("\n=== FINAL QUALITY ===")
print(json.dumps(result["report"]["final"], indent=2))
print("\n=== EXPORTED ARCHITECTURE ===")
print(json.dumps(result["metadata"], indent=2))

assert result["metadata"]["remaining_full_attention"] == {"encoder": 0, "decision_head": 0}
assert result["metadata"]["requires_laya"] is False


In [ ]:
#@title 4. Optional Hugging Face upload — held-out agreement gated
HF_REPO_ID = "vtavakkoli/Laya-PDelta3-GDN2-Standalone-Decision" #@param {type:"string"}
UPLOAD_TO_HF = False #@param {type:"boolean"}
MIN_AGREEMENT = 0.90 #@param {type:"number"}

if UPLOAD_TO_HF:
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        from huggingface_hub import notebook_login
        notebook_login()
    upload_export(result, HF_REPO_ID, token=token, min_agreement=MIN_AGREEMENT)
    print("uploaded:", f"https://huggingface.co/{HF_REPO_ID}")
else:
    print("Upload disabled; inspect final metrics first.")


In [ ]:
#@title 5. Reload WITHOUT Laya and use for game control
# Release the in-memory training student before constructing the reload copy.
result["student"].to("cpu")
if torch.cuda.is_available(): torch.cuda.empty_cache()

runtime = load_standalone_pdelta3(str(result["export_dir"]), device=result["device"])

game_state = {
    "speed": 42.0,
    "track_offset": -0.18,
    "heading_error": 0.07,
    "front_clearance": 18.5,
    "left_clearance": 7.0,
    "right_clearance": 4.2,
    "next_corner": "gentle_left",
}
actions = {
    "steer_left": "increase left steering while keeping safe speed",
    "straight": "hold current direction",
    "steer_right": "increase right steering",
    "brake": "reduce speed quickly",
    "accelerate": "increase throttle",
}
decision = runtime.decide(
    game_state,
    actions,
    instruction="Choose the safest useful control action for the next game step.",
)
print("choice:", decision.choice)
print("confidence:", round(decision.confidence, 4))
print(json.dumps(decision.probabilities, indent=2))
assert "laya" not in sys.modules
print("Standalone reload OK — no Laya dependency.")
